In [1]:
# Делаем ячейки пошире — просто удобнее читать
from IPython.display import HTML, display
display(HTML("<style>.container { width:90% !important; }</style>"))


# Работа с пропусками и переменными — простое объяснение

Это упрощённая версия практического урока. Здесь **нет «магии ML»**: мы шаг за шагом готовим таблицу данных так, чтобы компьютер мог по ней учиться.

## План

1. [Что такое предобработка данных](#prep)
2. [Какие шаги бывают](#sections)
3. [Переименование столбцов](#rename)
4. [Пропуски (дыры в данных)](#missing)
5. [Нормировка (привести числа к одной шкале)](#norm)
6. [Категориальные признаки (слова → числа)](#cat)
7. [Target Encoding (кодирование «по ответу»)](#te)
8. [Краткий итог](#итог)

---

### Словарь на старте (очень важно)

| Термин | Простыми словами |
|--------|------------------|
| **ML (Machine Learning)** | Машинное обучение: компьютер ищет закономерности в данных и учится предсказывать |
| **Датасет (dataset)** | Таблица с данными (как Excel-файл) |
| **Признак / переменная / feature** | Столбец таблицы (возраст, пол, цена…) |
| **Объект / строка / sample** | Одна запись — один человек, один товар, один день… |
| **Целевая переменная (target)** | То, что мы хотим предсказать (выжил / не выжил, цена, оценка…) |
| **Модель** | «Формула» или алгоритм, который по признакам угадывает target |
| **Предобработка** | Уборка и подготовка таблицы **до** обучения модели |


<a id="prep"></a>
## 1. Что такое предобработка данных?

Представьте, что вы хотите научить компьютер **предсказывать оценку ученика** по его возрасту, полу и любимому предмету.

Сырые данные часто «грязные»:
- где-то возраст не указан;
- пол записан то как «М», то как «male»;
- возраст — от 10 до 60, а «любимый предмет» — слова, а не числа;
- столбцы называются `x1`, `col_a` — непонятно, что это.

**Предобработка (data preprocessing / preparation)** — это:
1. **почистить** данные (дыры, ошибки, странные значения);
2. **привести к виду**, удобному для модели (числа, одна шкала, понятные названия).

> Важно: предобработку делают и для обучающих данных, и для новых (тех, на которых потом предсказываем). Иначе модель «увидит» другой формат и ошибётся.


<a id="sections"></a>
## 2. Какие шаги бывают?

### А. Трансформация (изменение вида данных)
- **Переименование** — дать столбцам понятные имена  
- **Кодирование категорий** — превратить «красный / синий» в числа  
- **Дискретизация (биннинг)** — возраст 23 → группа «молодой»  
- **Нормализация / стандартизация** — чтобы числа были сопоставимы  
- **Создание признаков** — например, «возраст в годах» → «возраст в группах»  

### Б. Интеграция
- Склеить данные из разных таблиц (оценки + посещаемость)

В этом ноутбуке сосредоточимся на самых частых: **пропуски, нормировка, категории**.


---
## Наши данные: маленький «школьный» датасет

Вместо большого Titanic возьмём **маленькую таблицу**, которую легко держать в голове.  
Это **учебный пример** — цифры выдуманы.

Столбцы:
- `name` — имя ученика  
- `age` — возраст (число; где-то может не быть)  
- `gender` — пол (слово)  
- `subject` — любимый предмет (слово)  
- `hours` — часов учёбы в неделю  
- `passed` — сдал ли экзамен: 1 = да, 0 = нет (**это target** — то, что предсказываем)


In [2]:
import pandas as pd
import numpy as np

# Маленькая таблица «вручную» — так проще понять, что происходит
df = pd.DataFrame({
    "name":    ["Аня", "Борис", "Вера", "Глеб", "Даша", "Егор", "Женя", "Зоя", "Илья", "Кира"],
    "age":     [15, 16, None, 17, 15, None, 18, 16, 14, 17],   # None = пропуск (дыры)
    "gender":  ["ж", "м", "ж", "м", "ж", "м", "ж", "ж", "м", "ж"],
    "subject": ["матем", "история", "матем", "физкультура", "матем", "история", "химия", "матем", "история", "химия"],
    "hours":   [10, 3, 12, 2, 8, 4, 15, 9, 5, 11],
    "passed":  [1, 0, 1, 0, 1, 0, 1, 1, 0, 1],
})

df


,name,age,gender,subject,hours,passed
0,Аня,15.0,ж,матем,10,1
1,Борис,16.0,м,история,3,0
2,Вера,NaN,ж,матем,12,1
3,Глеб,17.0,м,физкультура,2,0
4,Даша,15.0,ж,матем,8,1
5,Егор,NaN,м,история,4,0
6,Женя,18.0,ж,химия,15,1
7,Зоя,16.0,ж,матем,9,1
8,Илья,14.0,м,история,5,0
9,Кира,17.0,ж,химия,11,1


**Что видно сразу:**
- у Веры и Егора нет возраста (`NaN` — «Not a Number», то есть **пустое значение**);
- есть **числовые** столбцы (`age`, `hours`, `passed`);
- есть **категориальные** (словесные): `gender`, `subject`.

`pandas` — библиотека Python для таблиц.  
`NaN` / `None` / `null` — разные способы сказать «данных нет».


<a id="rename"></a>
## 3. Переименование

Имена столбцов должны быть **понятными** вам и коллегам.  
Плохо: `x1`, `col2`, `a`. Хорошо: `age`, `study_hours`, `exam_passed`.


In [3]:
# Переименуем для ясности (как будто пришли «кривые» имена)
df_renamed = df.rename(columns={
    "hours": "study_hours",   # сколько часов учился
    "passed": "exam_passed",  # сдал ли экзамен
})
df_renamed


,name,age,gender,subject,study_hours,exam_passed
0,Аня,15.0,ж,матем,10,1
1,Борис,16.0,м,история,3,0
2,Вера,NaN,ж,матем,12,1
3,Глеб,17.0,м,физкультура,2,0
4,Даша,15.0,ж,матем,8,1
5,Егор,NaN,м,история,4,0
6,Женя,18.0,ж,химия,15,1
7,Зоя,16.0,ж,матем,9,1
8,Илья,14.0,м,история,5,0
9,Кира,17.0,ж,химия,11,1


В остальной части урока оставим исходные короткие имена `hours` и `passed` — так компактнее.


<a id="missing"></a>
## 4. Пропуски — «дыры» в таблице

### Как выглядят пропуски
1. **Пустая ячейка**  
2. **Специальные метки**: `NaN`, `NA`, `null`, `None`  
3. **«Маскировочные» числа**: иногда пишут `-999` или `9999` вместо «неизвестно» (это **плохо**, модель может принять это за реальный возраст)

### Как найти пропуски


In [4]:
# True = в ячейке пропуск
df.isna()


,name,age,gender,subject,hours,passed
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,True,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
5,False,True,False,False,False,False
6,False,False,False,False,False,False
7,False,False,False,False,False,False
8,False,False,False,False,False,False
9,False,False,False,False,False,False


In [5]:
# Сколько пропусков в каждом столбце
df.isna().sum()


name       0
age        2
gender     0
subject    0
hours      0
passed     0
dtype: int64

In [6]:
# Сколько НЕ-пропусков в age
print("Пропусков в age:", df["age"].isna().sum())
print("Заполненных значений age:", df["age"].count())


Пропусков в age: 2
Заполненных значений age: 8


### Что можно сделать с пропусками?

| Способ | Идея | Когда уместно |
|--------|------|----------------|
| **Оставить** | Некоторые модели умеют работать с пропусками | Редко; зависит от алгоритма |
| **Удалить строки** | Выкинуть учеников без возраста | Если дыр мало |
| **Удалить столбец** | Выкинуть весь признак | Если дыр почти 100% |
| **Заполнить константой** | Подставить 0, −1, «неизвестно» | Осторожно: не создать «выброс» |
| **Заполнить средним / медианой / модой** | Типичное значение | Часто для чисел / категорий |
| **Восстановить моделью** | Угадать пропуск по другим признакам | Сложнее, для продвинутых |
| **Флаг «был пропуск»** | Добавить столбец `age_was_missing` | Полезно: сам факт пропуска может быть сигналом |

**Термины:**
- **Среднее (mean)** — сумма / количество. Чувствительно к «экстремальным» значениям.  
- **Медиана (median)** — значение «посередине», если отсортировать. Устойчивее к выбросам.  
- **Мода (mode)** — самое частое значение (удобно для слов: «матем»).  
- **Выброс (outlier)** — странное значение, сильно выбивающееся (возраст 200).


### Пример 1. Удалить строки с пропусками


In [7]:
# how='any' — удалить строку, если ХОТЬ в одном столбце есть пропуск
df_drop_rows = df.dropna(how="any")
print("Было строк:", len(df), "→ стало:", len(df_drop_rows))
df_drop_rows


Было строк: 10 → стало: 8


,name,age,gender,subject,hours,passed
0,Аня,15.0,ж,матем,10,1
1,Борис,16.0,м,история,3,0
3,Глеб,17.0,м,физкультура,2,0
4,Даша,15.0,ж,матем,8,1
6,Женя,18.0,ж,химия,15,1
7,Зоя,16.0,ж,матем,9,1
8,Илья,14.0,м,история,5,0
9,Кира,17.0,ж,химия,11,1


У нас всего 10 учеников, 2 без возраста — удалили 20% данных. На маленьких данных это больно. На больших иногда ок.


### Пример 2. Заполнить фиксированным значением


In [8]:
# Подставим -1 вместо пропуска (для демонстрации)
# На практике -1 для возраста — плохая идея: это «выброс»
df_fill_const = df.copy()
df_fill_const["age"] = df_fill_const["age"].fillna(-1)
df_fill_const[["name", "age"]]


,name,age
0,Аня,15.0
1,Борис,16.0
2,Вера,-1.0
3,Глеб,17.0
4,Даша,15.0
5,Егор,-1.0
6,Женя,18.0
7,Зоя,16.0
8,Илья,14.0
9,Кира,17.0


### Пример 3. Заполнить медианой (хороший базовый способ для чисел)


In [9]:
df_fill = df.copy()

median_age = df_fill["age"].median()  # медиана по тем, у кого возраст ЕСТЬ
print("Медиана возраста:", median_age)

df_fill["age"] = df_fill["age"].fillna(median_age)
df_fill[["name", "age"]]


Медиана возраста: 16.0


,name,age
0,Аня,15.0
1,Борис,16.0
2,Вера,16.0
3,Глеб,17.0
4,Даша,15.0
5,Егор,16.0
6,Женя,18.0
7,Зоя,16.0
8,Илья,14.0
9,Кира,17.0


### Пример 4. Добавить признак «был ли пропуск»

Идея: **сам факт**, что возраст не указали, может что-то значить (например, человек не заполнил анкету).  
Модель получит и «возраст (с подстановкой)», и «был ли пропуск».


In [10]:
df_flag = df.copy()

# 1 = возраст изначально был пустым, 0 = был заполнен
df_flag["age_was_missing"] = df_flag["age"].isna().astype(int)

# Затем заполняем медианой
df_flag["age"] = df_flag["age"].fillna(df_flag["age"].median())

df_flag[["name", "age", "age_was_missing"]]


,name,age,age_was_missing
0,Аня,15.0,0
1,Борис,16.0,0
2,Вера,16.0,1
3,Глеб,17.0,0
4,Даша,15.0,0
5,Егор,16.0,1
6,Женя,18.0,0
7,Зоя,16.0,0
8,Илья,14.0,0
9,Кира,17.0,0


### Пример 5. Мода для категориальных (если бы были пропуски в subject)

```python
# самое частое значение
most_common = df["subject"].mode()[0]
df["subject"] = df["subject"].fillna(most_common)
```

### Практические советы
1. **Сначала поймите почему** дыра: случайно забыли? человек отказался отвечать?  
2. Не заполняйте «до того, как сделали полезные новые признаки» вслепую — иногда порядок важен.  
3. `-999` как «заглушка» почти всегда хуже медианы/флага.

Дальше работаем с версией, где возраст заполнен медианой:


In [11]:
# Рабочая копия на весь остаток урока
work = df.copy()
work["age_was_missing"] = work["age"].isna().astype(int)
work["age"] = work["age"].fillna(work["age"].median())
work


,name,age,gender,subject,hours,passed,age_was_missing
0,Аня,15.0,ж,матем,10,1,0
1,Борис,16.0,м,история,3,0,0
2,Вера,16.0,ж,матем,12,1,1
3,Глеб,17.0,м,физкультура,2,0,0
4,Даша,15.0,ж,матем,8,1,0
5,Егор,16.0,м,история,4,0,1
6,Женя,18.0,ж,химия,15,1,0
7,Зоя,16.0,ж,матем,9,1,0
8,Илья,14.0,м,история,5,0,0
9,Кира,17.0,ж,химия,11,1,0


<a id="norm"></a>
## 5. Нормировка — «привести числа к одной шкале»

### Зачем?

Столбец `age` — примерно 14…18.  
Столбец `hours` — 2…15.  

А если бы был доход — 0…200 000?

Многие модели (особенно **линейная регрессия**, **KNN**, нейросети) «смотрят» на размер чисел.  
Признак с большими числами может **несправедливо перетянуть** внимание.

**Нормировка / нормализация** — преобразовать числа так, чтобы они стали **сопоставимы**.

> **Деревянные модели** (Random Forest, градиентный бустинг) к масштабу обычно **менее чувствительны**. Но нормировка всё равно часто полезна для единообразия пайплайна.


### 5.1. Min-Max: сжать в отрезок от 0 до 1

Формула:

$$
x_{new} = \frac{x - \min}{\max - \min}
$$

- самое маленькое значение → 0  
- самое большое → 1  
- остальное — между


In [12]:
work["age_minmax"] = (work["age"] - work["age"].min()) / (work["age"].max() - work["age"].min())
work["hours_minmax"] = (work["hours"] - work["hours"].min()) / (work["hours"].max() - work["hours"].min())

work[["name", "age", "age_minmax", "hours", "hours_minmax"]]


,name,age,age_minmax,hours,hours_minmax
0,Аня,15.0,0.25,10,0.615385
1,Борис,16.0,0.50,3,0.076923
2,Вера,16.0,0.50,12,0.769231
3,Глеб,17.0,0.75,2,0.000000
4,Даша,15.0,0.25,8,0.461538
5,Егор,16.0,0.50,4,0.153846
6,Женя,18.0,1.00,15,1.000000
7,Зоя,16.0,0.50,9,0.538462
8,Илья,14.0,0.00,5,0.230769
9,Кира,17.0,0.75,11,0.692308


### 5.2. Стандартизация (Z-score): среднее ≈ 0, «разброс» ≈ 1

Формула:

$$
x_{new} = \frac{x - \text{среднее}}{\text{стандартное отклонение}}
$$

**Стандартное отклонение (std)** — насколько значения «размазаны» вокруг среднего.  
После Z-score типичные значения лежат примерно от −2 до +2.


In [33]:
print(work["age"].std())
print(work["hours"].std())
work["age_zscore"] = (work["age"] - work["age"].mean()) / work["age"].std()
work["hours_zscore"] = (work["hours"] - work["hours"].mean()) / work["hours"].std()

work[["name", "age", "age_zscore", "hours", "hours_zscore"]]


1.1547005383792515
4.280446497997869


,name,age,age_zscore,hours,hours_zscore
0,Аня,15.0,-0.866025,10,0.490603
1,Борис,16.0,0.000000,3,-1.144740
2,Вера,16.0,0.000000,12,0.957844
3,Глеб,17.0,0.866025,2,-1.378361
4,Даша,15.0,-0.866025,8,0.023362
5,Егор,16.0,0.000000,4,-0.911120
6,Женя,18.0,1.732051,15,1.658705
7,Зоя,16.0,0.000000,9,0.256983
8,Илья,14.0,-1.732051,5,-0.677499
9,Кира,17.0,0.866025,11,0.724224


### 5.3. Деление на максимум

Самый простой вариант: `x / max(x)`.  
Тогда максимум станет 1, остальное — меньше 1.


In [14]:
work["age_div_max"] = work["age"] / work["age"].max()
work[["name", "age", "age_div_max"]]


,name,age,age_div_max
0,Аня,15.0,0.833333
1,Борис,16.0,0.888889
2,Вера,16.0,0.888889
3,Глеб,17.0,0.944444
4,Даша,15.0,0.833333
5,Егор,16.0,0.888889
6,Женя,18.0,1.000000
7,Зоя,16.0,0.888889
8,Илья,14.0,0.777778
9,Кира,17.0,0.944444


### 5.4. То же самое через sklearn (как делают «по-взрослому»)

`sklearn` — библиотека машинного обучения.  
`fit` = «запомнить параметры по данным» (min, max, mean, std).  
`transform` = «применить».


In [15]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max через sklearn
work["age_minmax_sk"] = MinMaxScaler().fit_transform(work[["age"]])

# Z-score через sklearn
work["age_zscore_sk"] = StandardScaler().fit_transform(work[["age"]])

work[["name", "age", "age_minmax", "age_minmax_sk", "age_zscore", "age_zscore_sk"]]


,name,age,age_minmax,age_minmax_sk,age_zscore,age_zscore_sk
0,Аня,15.0,0.25,0.25,-0.866025,-0.912871
1,Борис,16.0,0.50,0.50,0.000000,0.000000
2,Вера,16.0,0.50,0.50,0.000000,0.000000
3,Глеб,17.0,0.75,0.75,0.866025,0.912871
4,Даша,15.0,0.25,0.25,-0.866025,-0.912871
5,Егор,16.0,0.50,0.50,0.000000,0.000000
6,Женя,18.0,1.00,1.00,1.732051,1.825742
7,Зоя,16.0,0.50,0.50,0.000000,0.000000
8,Илья,14.0,0.00,0.00,-1.732051,-1.825742
9,Кира,17.0,0.75,0.75,0.866025,0.912871


### 5.5. Нормировка **внутри группы**

Иногда сравнивать «со всеми» бессмысленно.  
Пример: нормировать часы учёбы **отдельно среди сдавших и несдавших**.

`groupby` = разбить таблицу на группы.  
`transform` = посчитать что-то **внутри каждой группы** и вернуть столбец той же длины.


In [35]:
def z_score(x):
    # x — серия значений внутри одной группы
    return (x - x.mean()) / x.std()

work["hours_z_by_passed"] = work.groupby("passed")["hours"].transform(z_score)
work[["name", "passed", "hours", "hours_z_by_passed"]]


,name,passed,hours,hours_z_by_passed
0,Аня,1,10,-0.335578
1,Борис,0,3,-0.387298
2,Вера,1,12,0.469809
3,Глеб,0,2,-1.161895
4,Даша,1,8,-1.140965
5,Егор,0,4,0.387298
6,Женя,1,15,1.677890
7,Зоя,1,9,-0.738272
8,Илья,0,5,1.161895
9,Кира,1,11,0.067116


### 5.6. Ранговая нормировка

Идея: вместо «15 лет» хранить место в рейтинге (1-й, 2-й, …).  
Полезно, когда важнее порядок, а не точные числа.

- `average` — одинаковым значениям дают средний ранг  
- `min` — одинаковым значениям дают минимальный ранг  
- `max` — одинаковым значениям дают максимальный ранг  
- `dense` — ранги идут подряд без пропусков: 1, 2, 3, …  
- `ordinal` — просто порядковый номер после сортировки; у одинаковых значений порядок определяется по положению

In [36]:
from scipy.stats import rankdata

for method in ["average", "min", "max", "dense", "ordinal"]:
    work[f"age_rank_{method}"] = rankdata(work["age"], method=method)

# Можно ещё поделить на максимум ранга → снова шкала 0…1
work["age_rank_norm"] = work["age_rank_average"] / work["age_rank_average"].max()

work[["name", "age", "age_rank_average", "age_rank_min", "age_rank_max", "age_rank_dense", "age_rank_norm"]]


,name,age,age_rank_average,age_rank_min,age_rank_max,age_rank_dense,age_rank_norm
0,Аня,15.0,2.5,2,3,2,0.25
1,Борис,16.0,5.5,4,7,3,0.55
2,Вера,16.0,5.5,4,7,3,0.55
3,Глеб,17.0,8.5,8,9,4,0.85
4,Даша,15.0,2.5,2,3,2,0.25
5,Егор,16.0,5.5,4,7,3,0.55
6,Женя,18.0,10.0,10,10,5,1.00
7,Зоя,16.0,5.5,4,7,3,0.55
8,Илья,14.0,1.0,1,1,1,0.10
9,Кира,17.0,8.5,8,9,4,0.85


<a id="cat"></a>
## 6. Категориальные признаки: слова → числа

**Категориальный признак** — столбец, значения которого — **метки / классы**, а не «измеримые» числа.  
Примеры: цвет, город, пол, предмет, марка машины.

Модели почти всегда хотят **числа**. Значит, нужно **кодирование (encoding)**.

Два наших категориальных столбца: `gender` и `subject`.


In [18]:
print("Уникальные gender:", work["gender"].unique())
print("Уникальные subject:", work["subject"].unique())
work[["name", "gender", "subject", "passed"]]


Уникальные gender: ['ж' 'м']
Уникальные subject: ['матем' 'история' 'физкультура' 'химия']


,name,gender,subject,passed
0,Аня,ж,матем,1
1,Борис,м,история,0
2,Вера,ж,матем,1
3,Глеб,м,физкультура,0
4,Даша,ж,матем,1
5,Егор,м,история,0
6,Женя,ж,химия,1
7,Зоя,ж,матем,1
8,Илья,м,история,0
9,Кира,ж,химия,1


### 6.1. Label Encoding — «просто пронумеровать»

Каждой категории — свой номер: `ж→0`, `м→1` или `химия→0`, `матем→1`…

**Плюсы:** просто, мало памяти.  
**Минусы:**
- модель может подумать, что `2` «больше» `1` и «в два раза» (для **линейных** моделей это плохо);
- если появится **новая** категория в будущем — непонятно, какой номер дать.

Способы:
- **LabelEncoder (sklearn)** — обычно в **алфавитном** (лексикографическом) порядке  
- **factorize (pandas)** — в **порядке первого появления** в таблице  
- **случайные числа** — почти никогда не нужно, только для экспериментов


In [19]:
from sklearn.preprocessing import LabelEncoder

demo = work[["name", "gender", "subject"]].copy()

le = LabelEncoder()
# Алфавитный порядок: «ж» и «м» → 0 и 1 (зависит от сортировки меток)
demo["gender_le"] = le.fit_transform(demo["gender"])

# В порядке появления в таблице
demo["gender_fz"] = pd.factorize(demo["gender"])[0]
demo["subject_fz"] = pd.factorize(demo["subject"])[0]

# Случайный словарь (для демонстрации идеи)
rng = np.random.default_rng(42)  # фиксируем «зерно», чтобы результат повторялся
subjects = demo["subject"].unique()
rnd_map = dict(zip(subjects, rng.random(len(subjects)).round(2)))
print("Случайный словарь:", rnd_map)
demo["subject_rnd"] = demo["subject"].map(rnd_map)

demo


Случайный словарь: {'матем': np.float64(0.77), 'история': np.float64(0.44), 'физкультура': np.float64(0.86), 'химия': np.float64(0.7)}


,name,gender,subject,gender_le,gender_fz,subject_fz,subject_rnd
0,Аня,ж,матем,0,0,0,0.77
1,Борис,м,история,1,1,1,0.44
2,Вера,ж,матем,0,0,0,0.77
3,Глеб,м,физкультура,1,1,2,0.86
4,Даша,ж,матем,0,0,0,0.77
5,Егор,м,история,1,1,1,0.44
6,Женя,ж,химия,0,0,3,0.70
7,Зоя,ж,матем,0,0,0,0.77
8,Илья,м,история,1,1,1,0.44
9,Кира,ж,химия,0,0,3,0.70


### 6.2. One-Hot Encoding (Dummy) — «столбец на каждую категорию»

Идея: вместо одного столбца `subject` сделать несколько **бинарных** (0/1):

| subject | subject_матем | subject_история | subject_химия | subject_физкультура |
|---------|---------------|-----------------|---------------|---------------------|
| матем   | 1             | 0               | 0             | 0                   |
| история | 0             | 1               | 0             | 0                   |

**Плюсы:** нет ложного «порядка» между категориями — хорошо для линейных моделей.  
**Минусы:**
- если категорий **очень много** (1000 городов) — столбцов будет 1000 (**разреженная** таблица: почти все нули);
- деревья иногда работают чуть хуже на куче бинарных колонок.

**Бинарный** = принимает только два значения (0 или 1).


In [20]:
# Самый простой способ в pandas
ohe = pd.get_dummies(work[["name", "subject"]], columns=["subject"], dtype=int)
ohe


,name,subject_история,subject_матем,subject_физкультура,subject_химия
0,Аня,0,1,0,0
1,Борис,1,0,0,0
2,Вера,0,1,0,0
3,Глеб,0,0,1,0
4,Даша,0,1,0,0
5,Егор,1,0,0,0
6,Женя,0,0,0,1
7,Зоя,0,1,0,0
8,Илья,1,0,0,0
9,Кира,0,0,0,1


In [21]:
# Через sklearn OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)  # обычный плотный массив, не «сжатый»
arr = encoder.fit_transform(work[["subject"]])

# Имена столбцов — сами категории
ohe_cols = [f"subject={c}" for c in encoder.categories_[0]]
ohe_df = pd.DataFrame(arr, columns=ohe_cols, index=work.index)

pd.concat([work[["name", "subject"]], ohe_df.astype(int)], axis=1)


,name,subject,subject=история,subject=матем,subject=физкультура,subject=химия
0,Аня,матем,0,1,0,0
1,Борис,история,1,0,0,0
2,Вера,матем,0,1,0,0
3,Глеб,физкультура,0,0,1,0
4,Даша,матем,0,1,0,0
5,Егор,история,1,0,0,0
6,Женя,химия,0,0,0,1
7,Зоя,матем,0,1,0,0
8,Илья,история,1,0,0,0
9,Кира,химия,0,0,0,1


### Ручной One-Hot (чтобы понять «магию»)


In [22]:
manual = work[["name", "subject"]].copy()

for value in manual["subject"].unique():
    # 1, если subject равен value, иначе 0
    manual[f"subject={value}"] = (manual["subject"] == value).astype(int)

manual


,name,subject,subject=матем,subject=история,subject=физкультура,subject=химия
0,Аня,матем,1,0,0,0
1,Борис,история,0,1,0,0
2,Вера,матем,1,0,0,0
3,Глеб,физкультура,0,0,1,0
4,Даша,матем,1,0,0,0
5,Егор,история,0,1,0,0
6,Женя,химия,0,0,0,1
7,Зоя,матем,1,0,0,0
8,Илья,история,0,1,0,0
9,Кира,химия,0,0,0,1


### 6.3. Дискретизация (биннинг) — число → «корзина»

**Биннинг (binning / discretization)** — разрезать непрерывное число на **интервалы** и дать им метки.

Пример: возраст  
- 0–12 → ребёнок  
- 12–15 → подросток  
- 15–17 → старший школьник  
- 17+ → почти взрослый  

**Плюсы:** проще объяснить человеку; иногда помогает простым моделям.  
**Минусы:** теряем точность (16 и 16.9 стали «одинаковыми»).


In [23]:
# Границы «корзин» и подписи
bins = [0, 15, 16, 17, 100]
labels = ["до_15", "15_16", "16_17", "17_плюс"]

work["age_group"] = pd.cut(
    work["age"],
    bins=bins,
    labels=labels,
    include_lowest=True,  # включить левую границу
)

work[["name", "age", "age_group"]]


,name,age,age_group
0,Аня,15.0,до_15
1,Борис,16.0,15_16
2,Вера,16.0,15_16
3,Глеб,17.0,16_17
4,Даша,15.0,до_15
5,Егор,16.0,15_16
6,Женя,18.0,17_плюс
7,Зоя,16.0,15_16
8,Илья,14.0,до_15
9,Кира,17.0,16_17


#### Два популярных автоматических способа

1. **Equal-width** — интервалы **равной длины** (`pd.cut`)  
   Пример: от min до max разрезать на 4 равных отрезка.

2. **Equal-frequency (квантили)** — в каждый интервал попадает **примерно одинаковое число людей** (`pd.qcut`)


In [24]:
# 3 интервала равной ширины
work["age_eq_width"] = pd.cut(work["age"], bins=3)

# 3 интервала с примерно равным числом людей
# duplicates='drop' — на случай, если квантили совпадут (мало уникальных возрастов)
work["age_eq_freq"] = pd.qcut(work["age"], q=3, duplicates="drop")

work[["name", "age", "age_eq_width", "age_eq_freq"]]


,name,age,age_eq_width,age_eq_freq
0,Аня,15.0,"(13.996, 15.333]","(13.999, 16.0]"
1,Борис,16.0,"(15.333, 16.667]","(13.999, 16.0]"
2,Вера,16.0,"(15.333, 16.667]","(13.999, 16.0]"
3,Глеб,17.0,"(16.667, 18.0]","(16.0, 18.0]"
4,Даша,15.0,"(13.996, 15.333]","(13.999, 16.0]"
5,Егор,16.0,"(15.333, 16.667]","(13.999, 16.0]"
6,Женя,18.0,"(16.667, 18.0]","(16.0, 18.0]"
7,Зоя,16.0,"(15.333, 16.667]","(13.999, 16.0]"
8,Илья,14.0,"(13.996, 15.333]","(13.999, 16.0]"
9,Кира,17.0,"(16.667, 18.0]","(16.0, 18.0]"


### 6.4. Хэш-кодирование — «сжать» кучу категорий в несколько чисел

Представьте 10 000 разных городов. One-Hot даст 10 000 столбцов — тяжело.

**Хэш-функция** — «перемешиватель», который из слова делает число (как отпечаток).  
Мы просим: «положи все категории в **K** корзин» (например, K=4).

**Плюс:** мало столбцов.  
**Минус:** **коллизии** — разные слова могут попасть в одну корзину (как два человека с одним отпечатком пальца… почти).


In [25]:
from sklearn.feature_extraction import FeatureHasher

# n_features=3 → всего 3 числовых столбца вместо N категорий
fh = FeatureHasher(n_features=3, input_type="string")

# FeatureHasher ждёт список токенов на каждую строку
values = work["subject"].astype(str).apply(lambda x: [x])
hashed = fh.transform(values).toarray()

hash_df = pd.DataFrame(
    hashed,
    columns=["subject_hash_0", "subject_hash_1", "subject_hash_2"],
    index=work.index,
)

pd.concat([work[["name", "subject"]], hash_df], axis=1)


,name,subject,subject_hash_0,subject_hash_1,subject_hash_2
0,Аня,матем,0.0,-1.0,0.0
1,Борис,история,0.0,1.0,0.0
2,Вера,матем,0.0,-1.0,0.0
3,Глеб,физкультура,-1.0,0.0,0.0
4,Даша,матем,0.0,-1.0,0.0
5,Егор,история,0.0,1.0,0.0
6,Женя,химия,0.0,-1.0,0.0
7,Зоя,матем,0.0,-1.0,0.0
8,Илья,история,0.0,1.0,0.0
9,Кира,химия,0.0,-1.0,0.0


### Проблема редких и новых категорий

- Категория встретилась **1 раз** — по ней опасно «учить» (случайность).  
  Часто **склеивают** редкие в одну метку `other`.  
- Категория **не встречалась при обучении**, а появилась потом — модель не знает, что с ней делать.  
  Нужна стратегия: `other`, хэш, или модель, умеющая это (CatBoost и др.).


<a id="te"></a>
## 7. Target Encoding — кодировать категорию «по среднему ответу»

**Идея:** заменить категорию **средним значением target** для этой категории.

Пример с `subject` и `passed` (1 = сдал, 0 = нет):

| subject | доля сдавших (примерно) |
|---------|-------------------------|
| матем   | высокая                 |
| история | низкая                  |

Тогда `матем → 0.9`, `история → 0.2` — это уже **числа с смыслом**.

**Опасности:**
1. **Утечка target (target leakage)** — если неаккуратно подсмотреть «ответ» из той же строки, модель «спишет» и на новых данных провалится.  
2. **Редкие категории** — если предмет встретился 1 раз и человек сдал, среднее = 1.0 — **переобучение** (модель запомнила случайность).

**Сглаживание (smoothing):** смешиваем среднее по категории со **средним по всем**, чтобы редкие не «улетали».

Упрощённо:

$$
\text{код} = \frac{n \cdot \text{среднее\_категории} + \alpha \cdot \text{среднее\_всего}}{n + \alpha}
$$

где $n$ — сколько раз встретилась категория, $\alpha$ — сила «доверия» к общему среднему.


In [26]:
# Простой target encoding БЕЗ сглаживания (только для понимания!)
# На практике для обучения модели делают аккуратнее (кросс-валидация / leave-one-out)

global_mean = work["passed"].mean()
print("Общая доля сдавших:", round(global_mean, 3))

# Средний passed внутри каждого subject
subject_mean = work.groupby("subject")["passed"].mean()
print("\nСредний passed по предметам:")
print(subject_mean)

work["subject_te"] = work["subject"].map(subject_mean)
work[["name", "subject", "passed", "subject_te"]]


Общая доля сдавших: 0.6

Средний passed по предметам:
subject
история        0.0
матем          1.0
физкультура    0.0
химия          1.0
Name: passed, dtype: float64


,name,subject,passed,subject_te
0,Аня,матем,1,1.0
1,Борис,история,0,0.0
2,Вера,матем,1,1.0
3,Глеб,физкультура,0,0.0
4,Даша,матем,1,1.0
5,Егор,история,0,0.0
6,Женя,химия,1,1.0
7,Зоя,матем,1,1.0
8,Илья,история,0,0.0
9,Кира,химия,1,1.0


### Сглаживание на пальцах


In [27]:
alpha = 3  # чем больше alpha, тем сильнее тянем к общему среднему

stats = work.groupby("subject")["passed"].agg(["mean", "count"])
stats.columns = ["cat_mean", "n"]

# Сглаженное значение
stats["smooth"] = (stats["n"] * stats["cat_mean"] + alpha * global_mean) / (stats["n"] + alpha)
print(stats)

work["subject_te_smooth"] = work["subject"].map(stats["smooth"])
work[["name", "subject", "passed", "subject_te", "subject_te_smooth"]]


             cat_mean  n    smooth
subject                           
история           0.0  3  0.300000
матем             1.0  4  0.828571
физкультура       0.0  1  0.450000
химия             1.0  2  0.760000


,name,subject,passed,subject_te,subject_te_smooth
0,Аня,матем,1,1.0,0.828571
1,Борис,история,0,0.0,0.300000
2,Вера,матем,1,1.0,0.828571
3,Глеб,физкультура,0,0.0,0.450000
4,Даша,матем,1,1.0,0.828571
5,Егор,история,0,0.0,0.300000
6,Женя,химия,1,1.0,0.760000
7,Зоя,матем,1,1.0,0.828571
8,Илья,история,0,0.0,0.300000
9,Кира,химия,1,1.0,0.760000


Заметьте: для редких предметов `smooth` ближе к общему среднему, чем «сырое» среднее — так и задумано.


### Добавление шума

Иногда к target encoding **добавляют небольшой шум** (случайные колебания), чтобы модель меньше «зазубривала» точные числа.

Это **регуляризация** — способ сделать модель менее «уверенной» в случайных совпадениях.


In [28]:
def add_noise(series, noise_level=0.05, seed=0):
    rng = np.random.default_rng(seed)
    # умножаем на (1 + маленький случайный процент)
    return series * (1 + noise_level * rng.standard_normal(len(series)))

work["subject_te_noisy"] = add_noise(work["subject_te_smooth"], noise_level=0.05)
work[["name", "subject_te_smooth", "subject_te_noisy"]]


,name,subject_te_smooth,subject_te_noisy
0,Аня,0.828571,0.833780
1,Борис,0.300000,0.298018
2,Вера,0.828571,0.855103
3,Глеб,0.450000,0.452360
4,Даша,0.828571,0.806379
5,Егор,0.300000,0.305424
6,Женя,0.760000,0.809552
7,Зоя,0.828571,0.867808
8,Илья,0.300000,0.289444
9,Кира,0.760000,0.711914


### Кодирование «как в CatBoost» (по предыдущим объектам)

Идея **leave-one-out / expanding mean**:
для каждой строки среднее по категории считается **без учёта самой этой строки** (или только по «предыдущим»).

Так меньше утечки: строка не голосует сама за себя.


In [29]:
# Expanding mean внутри группы subject:
# (сумма passed до текущей строки включительно - текущий passed) / (число строк до текущей)
work["subject_cb"] = (
    (work.groupby("subject")["passed"].cumsum() - work["passed"])
    / work.groupby("subject").cumcount()
)

# У первой строки каждой группы деление на 0 → NaN → заполним общим средним
work["subject_cb"] = work["subject_cb"].fillna(global_mean)

work[["name", "subject", "passed", "subject_te", "subject_cb"]]


,name,subject,passed,subject_te,subject_cb
0,Аня,матем,1,1.0,0.6
1,Борис,история,0,0.0,0.6
2,Вера,матем,1,1.0,1.0
3,Глеб,физкультура,0,0.0,0.6
4,Даша,матем,1,1.0,1.0
5,Егор,история,0,0.0,0.0
6,Женя,химия,1,1.0,0.6
7,Зоя,матем,1,1.0,1.0
8,Илья,история,0,0.0,0.0
9,Кира,химия,1,1.0,1.0


### Другие статистики (идея)

Если target бинарный (0/1), вместо среднего можно кодировать:
- разницу «успехов и неуспехов»;
- логарифмы счётчиков;
- нормализованную разницу.

Смысл тот же: **сжать категорию в число, связанное с ответом**.  
Для старта достаточно **среднего со сглаживанием**.


---
## Мини-конвейер: от «сырой» таблицы к «готовой»

Соберём всё вместе на **чистой копии** исходных данных — как чеклист перед моделью.


In [30]:
raw = df.copy()  # исходная таблица с пропусками

# 1) Флаг пропуска + заполнение медианой
raw["age_was_missing"] = raw["age"].isna().astype(int)
raw["age"] = raw["age"].fillna(raw["age"].median())

# 2) Нормировка чисел (min-max)
raw["age_scaled"] = MinMaxScaler().fit_transform(raw[["age"]])
raw["hours_scaled"] = MinMaxScaler().fit_transform(raw[["hours"]])

# 3) One-Hot для gender и subject
raw = pd.get_dummies(raw, columns=["gender", "subject"], dtype=int)

# 4) Имя для модели не нужно (это идентификатор, не закономерность)
X = raw.drop(columns=["name", "passed", "age", "hours"])  # признаки
y = raw["passed"]  # target

print("Признаки (X), на которых училась бы модель:")
from IPython.display import display
display(X)
print("\nTarget (y):")
print(y.tolist())
print("\nФорма X:", X.shape, "— строк:", X.shape[0], ", столбцов-признаков:", X.shape[1])


Признаки (X), на которых училась бы модель:


,age_was_missing,age_scaled,hours_scaled,gender_ж,gender_м,subject_история,subject_матем,subject_физкультура,subject_химия
0,0,0.25,0.615385,1,0,0,1,0,0
1,0,0.50,0.076923,0,1,1,0,0,0
2,1,0.50,0.769231,1,0,0,1,0,0
3,0,0.75,0.000000,0,1,0,0,1,0
4,0,0.25,0.461538,1,0,0,1,0,0
5,1,0.50,0.153846,0,1,1,0,0,0
6,0,1.00,1.000000,1,0,0,0,0,1
7,0,0.50,0.538462,1,0,0,1,0,0
8,0,0.00,0.230769,0,1,1,0,0,0
9,0,0.75,0.692308,1,0,0,0,0,1



Target (y):
[1, 0, 1, 0, 1, 0, 1, 1, 0, 1]

Форма X: (10, 9) — строк: 10 , столбцов-признаков: 9


**Что получилось:**
- дыр нет;
- числа в сопоставимой шкале;
- слова превратились в 0/1 столбцы;
- `y` — то, что предсказываем.

Дальше уже можно вызывать `model.fit(X, y)` — но это тема следующих уроков.


<a id="итог"></a>
## 8. Итог простыми словами

1. **Предобработка** — уборка и перевод данных в формат, понятный модели. Часто занимает **большую часть** времени проекта.  
2. **Пропуски** — не «просто нули». Лучше: понять причину → заполнить умно (медиана/мода) и/или добавить флаг «был пропуск». Удаление — только если дыр мало.  
3. **Нормировка** — чтобы признаки с разными шкалами не перетягивали одеяло. Min-Max → [0, 1], Z-score → среднее 0.  
4. **Категории** нельзя скормить модели «как текст» (обычно).  
   - мало уникальных значений → **One-Hot**;  
   - много → хэш / target encoding / спец. модели;  
   - Label Encoding — осторожно с линейными моделями.  
5. **Target Encoding** мощный, но опасен **утечкой** и переобучением на редких категориях → сглаживание, шум, leave-one-out.  
6. Главный вопрос перед любым трюком: **«Почему в данных так?»**

### Шпаргалка «что делать»

| Ситуация | Что попробовать |
|----------|-----------------|
| Число с дырами | медиана + флаг пропуска |
| Категория с дырами | мода или отдельная метка `"unknown"` |
| Числа в разных шкалах | Min-Max или StandardScaler |
| Категория, 2–20 значений | One-Hot (`get_dummies`) |
| Категория, сотни/тысячи значений | хэш, target encoding, CatBoost |
| Возраст/доход хочется упростить | биннинг (`cut` / `qcut`) |

### Дополнительные материалы (на английском, с Kaggle)

- [9 ways to treat categorical features](https://www.kaggle.com/mlisovyi/9-ways-to-treat-categorical-features-updated#)  
- [Target encoding for categorical features](https://www.kaggle.com/ogrellier/python-target-encoding-for-categorical-features#)  
- [Mean likelihood encodings](https://www.kaggle.com/vprokopev/mean-likelihood-encodings-a-comprehensive-study#)


---
### Мини-глоссарий

| Термин | Смысл |
|--------|--------|
| **Feature / признак** | Входной столбец |
| **Target** | Что предсказываем |
| **NaN** | Пропуск |
| **Imputation** | Заполнение пропусков |
| **Scaling / Normalization** | Приведение к шкале |
| **Encoding** | Кодирование категорий в числа |
| **One-Hot** | Отдельный 0/1 столбец на категорию |
| **Label Encoding** | Пронумеровать категории |
| **Binning** | Нарезать числа на интервалы |
| **Overfitting** | Модель зазубрила обучающие данные и плохо работает на новых |
| **Leakage** | Случайно «подсмотрели» ответ при подготовке признаков |
| **Pipeline** | Конвейер шагов обработки |
